# Shipping and Maintaining Optimised Prompts [Agent Patterns - Module 09]

> **MLCourse - Agentic AI - Agent Patterns**

Notebook 03 produced `triage_optimized.json`. That file is now a build
artifact of your system, with the same lifecycle questions as any other:
where does it live, what invalidates it, and how do you know when it has
gone stale?

### What you will learn

1. Loading a compiled program back and reusing it.
2. Composing a DSPy module into a larger program.
3. Cost accounting: what compiling actually spent.
4. The operational checklist for optimised prompts.

### Key takeaways

- A compiled program is a versioned artifact, not a cached side effect.
- Model swap, data drift, or rubric change all invalidate it.
- Keep the eval set in the repo. It is the only thing that catches drift.

### Setup: imports, environment, track discovery


In [ ]:
import os
import time
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq)")
print(f"Key loaded : {bool(GROQ_API_KEY)}")


### Point DSPy at Groq (through LiteLLM)


In [ ]:
# dspy.LM is a thin wrapper over LiteLLM. The "groq/" prefix is the LiteLLM
# provider route - the rest is the Groq model id.

import dspy

lm = dspy.LM(
    f"groq/{MODEL}",
    api_key=GROQ_API_KEY,
    temperature=0.0,      # deterministic-ish: we are going to MEASURE things
    max_tokens=700,
    num_retries=5,        # LiteLLM backs off on 429 (Groq free tier = 8000 TPM)
)
dspy.configure(lm=lm)

print("DSPy configured.")
print("dspy version:", dspy.__version__)


### Load the artifact compiled in notebook 03


In [ ]:
from typing import Literal

class Triage(dspy.Signature):
    """Assign a support priority to an incoming customer message."""
    message: str = dspy.InputField(desc="the raw customer message")
    priority: Literal["P0", "P1", "P2"] = dspy.OutputField(desc="the triage priority")

ARTIFACT = Path("triage_optimized.json")
print("artifact exists:", ARTIFACT.exists())

triage = dspy.Predict(Triage)
triage.load(ARTIFACT)          # signature must match what was compiled
print("loaded demos:", len(triage.demos))


Note the constraint: **you must reconstruct the same signature class before
loading.** The JSON stores demos and instructions, not the Python type. So
the signature definition and the artifact must be versioned together - if
someone renames `priority` to `level`, the artifact is silently wrong.

Put them in the same module, and treat a signature edit as a re-compile
trigger.

### Use it


In [ ]:
NEW = [
    "The checkout page has been throwing errors for two hours, no orders.",
    "I see a duplicate charge from last Tuesday on my statement.",
    "Small suggestion: could the sidebar be collapsible?",
]

for m in NEW:
    out = triage(message=m)
    print(f"{out.priority}  {m}")
    time.sleep(1.5)


### 1. Composing modules

A DSPy `Module` is just a class with `forward()`. Inside it you call other
modules. Each sub-module has its own signature and can be optimised
independently - or jointly, since the optimiser walks the whole tree.

Here: triage first, then draft a reply whose tone depends on the priority.
Two LLM calls, one composed object.

### A two-step composed module


In [ ]:
class Reply(dspy.Signature):
    """Draft a short first-response to a customer, matching the urgency."""
    message: str = dspy.InputField()
    priority: str = dspy.InputField(desc="P0 (urgent), P1 (money), P2 (routine)")
    reply: str = dspy.OutputField(desc="two sentences maximum, no greeting")

class SupportDesk(dspy.Module):
    def __init__(self, triage_module):
        super().__init__()
        self.triage = triage_module           # already compiled
        self.reply = dspy.Predict(Reply)      # not compiled yet

    def forward(self, message):
        t = self.triage(message=message)
        r = self.reply(message=message, priority=t.priority)
        return dspy.Prediction(priority=t.priority, reply=r.reply)

desk = SupportDesk(triage)

out = desk(message="Nobody on my team can log in, we are completely blocked.")
print("priority:", out.priority)
print("reply   :", out.reply.strip())


### Sub-modules are individually addressable


In [ ]:
# This is why composition matters: you can compile the weak link only.

for name, sub in desk.named_predictors():
    n_demos = len(getattr(sub, "demos", []))
    print(f"  {name:12s} demos={n_demos}  signature={sub.signature.__name__}")


`triage` carries demos (compiled); `reply` carries none (raw). In a real
system you would compile whichever one your evals show is failing - not
both, because each compile costs tokens.

### 2. What did it cost?

DSPy tracks usage on the LM object. Compiling is not free and you should be
able to state the number, not hand-wave it.

### Token accounting


In [ ]:
hist = lm.history
total_in = sum(h.get("usage", {}).get("prompt_tokens", 0) or 0 for h in hist)
total_out = sum(h.get("usage", {}).get("completion_tokens", 0) or 0 for h in hist)

print(f"calls in this notebook : {len(hist)}")
print(f"prompt tokens          : {total_in}")
print(f"completion tokens      : {total_out}")
print(f"total                  : {total_in + total_out}")
print()
print("Compare against Groq free tier: 8000 tokens/minute.")
print("A BootstrapFewShot compile over 8 examples is small. MIPROv2 over")
print("200 examples is not - always check the optimiser's call count first.")


### 3. Operational checklist

**Version together.** Signature definition + compiled JSON + eval set live
in the same directory and move in the same commit.

**Re-compile when:**
- the model or model version changes,
- the signature changes (fields renamed, types tightened),
- the rubric/policy changes,
- the eval score drops below your threshold.

**Keep the eval set in the repo.** It is cheap, it is the only thing that
detects drift, and it doubles as executable documentation of the rubric.

**Run the eval in CI**, on a fixed seed and `temperature=0`. Fail the build
if accuracy drops more than a set margin. This is the whole point of having
a number instead of a vibe.

**Do not hand-edit the compiled JSON.** If you find yourself wanting to,
that is a signal your examples are wrong - fix the data and re-compile.

### Pitfalls recap

- **Silent signature drift** is the top failure mode. Renaming a field
  without re-compiling loads demos that no longer match.
- **Optimising a metric you do not trust** produces confident garbage.
  Spend effort on the metric before the optimiser.
- **Compiling on every process start** wastes tokens and makes startup
  nondeterministic. Compile offline, commit the artifact, load at runtime.
- **Assuming portability.** Demos tuned on one model can underperform on
  another. Re-run the eval after any model change.

### Module wrap-up

You measured a real before/after on a real Groq model. The technique
itself is unremarkable - a filtered few-shot search - but the *discipline*
around it is the transferable part:

1. Write examples that encode your conventions.
2. Write a metric you would defend in review.
3. Measure the baseline.
4. Optimise.
5. Measure again, on held-out data, and report the number either way.

That loop works with DSPy and it works without it.